### html + htmlheadertextsplitter+ remove html junk + better chunking
### all-MiniLM-L6-v2 + larger chunk size + excluding sources

In [7]:
import sys
import os
import chromadb
path = "C:\\Users\\luyer\\my_rag\\data"
client = chromadb.PersistentClient(path=path)

In [8]:
from datetime import datetime, timezone

In [ ]:
# for collection in client.list_collections():
#     client.delete_collection(name=collection.name)

In [9]:
for item in client.list_collections():
    # Chroma versions differ: list_collections() may return names or objects
    name = item if isinstance(item, str) else item.name
    collection = client.get_collection(name=name)

    print(f"\nCollection: {collection.name}")
    print(f"ID: {collection.id}")
    print(f"Count: {collection.count()}")
    print(f"Metadata: {collection.metadata}")
    print(f"Configuration: {getattr(collection, 'configuration', None)}")


Collection: my_movies_html_config_cosine
ID: 51ca53f2-f0af-4252-8182-c0f9cf14b099
Count: 1471
Metadata: {'source': 'file_indexing', 'created_at': '2026-10-22T01:10:41.387929+00:00', 'version': 1}
Configuration: {'hnsw': {'space': 'cosine', 'ef_construction': 200, 'ef_search': 200, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': <chromadb.api.types.DefaultEmbeddingFunction object at 0x000001BC7FCCE570>}

Collection: my_movies_html_config_cosine_10262026
ID: 60d90cf2-164f-4b9c-8b4c-55669c9c6994
Count: 0
Metadata: {'created_at': '2026-10-27T03:11:32.602065+00:00', 'source': 'file_indexing', 'version': 1}
Configuration: {'hnsw': {'space': 'cosine', 'ef_construction': 200, 'ef_search': 200, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': <chromadb.api.types.DefaultEmbeddingFunction object at 0x000001BC7FD59A50>}


In [10]:
#Fix HTML extraction and rebuild the collection. Preserve linked words in their original sentence positions. Check the Kansas City and spring 1962 passages as small acceptance examples before reindexing everything. Also check UTF-8 handling; the traces show garbled accented characters
cosine_collection = client.get_or_create_collection(
    name="my_movies_html_config_cosine_10262026",
    configuration={
        "hnsw": {"space": "cosine",
                 "ef_construction": 200,
                 "ef_search": 200
                 }
    },
    metadata={
        "created_at": datetime.now(timezone.utc).isoformat(),
        "source": "file_indexing",
        "version": 1,
    }
)

In [11]:
files = [
    file_name
    for file_name in os.listdir(path)
    if os.path.isfile(os.path.join(path, file_name)) and file_name.endswith('.html')
]
print(files)
print(len(files))

['2001_A_Space_Odyssey.html', '8½.html', 'Casablanca.html', 'Citizen_Kane.html', 'Gone_with_the_Wind.html', 'His_Girl_Friday.html', 'Parasite_2019.html', 'Pulp_Fiction.html', 'Seven_Samurai.html', 'Spirited_Away.html', 'The_Dark_Knight.html', 'The_Godfather.html', 'The_Matrix.html', 'The_Narrow_Margin.html']
14


### Extraction and splitter settings

In [12]:
import re
from pathlib import Path
from langchain_text_splitters import (
    HTMLHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

def clean_chunk_text(text: str) -> str:
    """Clean text after HTML parsing."""
    text = text.replace("\x00", "")

    # remove Wikipedia citation markers like [1], [22], [123]
    text = re.sub(r"\[\d+\]", "", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

header_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=[
        ("h1", "header1"),
        ("h2", "header2"),
        ("h3", "header3"),
    ]
)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "? ",
        "! ",
        " ",
        "",
    ]
)

BAD_SECTIONS = {
    "References",
    "External links",
    "Further reading",
    "Sources"
}

import sys
extraction_dir = Path(path).resolve().parent / "index"
if str(extraction_dir) not in sys.path:
    sys.path.insert(0, str(extraction_dir))
from html_extraction import prepare_article_html, movie_title, is_heading_only


### Acceptance checks before corpus preparation

In [13]:
def check_extraction_acceptance():
    checks = [
        ("The_Godfather.html", "set in contemporary Kansas City and shot in the studio backlot"),
        ("8½.html", "cast Mastroianni, Anouk Aimée, and Sandra Milo in lead roles"),
    ]
    for name, expected in checks:
        raw = (Path(path) / name).read_text(encoding="utf-8", errors="strict")
        sections = header_splitter.split_text(prepare_article_html(raw))
        text = clean_chunk_text(" ".join(doc.page_content for doc in sections))
        assert expected in text, f"Extraction acceptance failed: {name}"
        chunks = text_splitter.split_documents(sections)
        assert any(expected in clean_chunk_text(doc.page_content) for doc in chunks), name
        print(f"PASS {name}: {expected}")
    print("UTF-8 accented-name and inline-link acceptance checks passed.")

# Fail before processing the corpus or adding documents if extraction regresses.
check_extraction_acceptance()



PASS The_Godfather.html: set in contemporary Kansas City and shot in the studio backlot
PASS 8½.html: cast Mastroianni, Anouk Aimée, and Sandra Milo in lead roles
UTF-8 accented-name and inline-link acceptance checks passed.


### Filter, deduplicate, and enrich movie chunks

In [14]:
def build_movie_chunks(file_name):
    # Decode strictly so invalid UTF-8 raises an error rather than losing characters.
    raw = (Path(path) / file_name).read_text(encoding="utf-8", errors="strict")
    # Resolve the movie title for embedded context and metadata filtering.
    title = movie_title(raw, file_name)
    # Preserve inline word order, then group content by HTML headings.
    sections = header_splitter.split_text(prepare_article_html(raw))
    # Count discarded headings and repeated chunks for the notebook summary.
    removed = {"heading_only": 0, "duplicates": 0}
    substantive = []
    for doc in sections:
        # Exclude configured sections such as References and External links.
        if doc.metadata.get("header2") in BAD_SECTIONS:
            continue
        # Drop standalone labels like Plot that contain no article content.
        if is_heading_only(doc.page_content, doc.metadata):
            removed["heading_only"] += 1
            continue
        substantive.append(doc)
    # Reset duplicate tracking per movie to retain cross-movie provenance.
    seen = set()
    items = []
    # Split substantive sections into overlapping chunks using configured sizes.
    for chunk_index, chunk in enumerate(text_splitter.split_documents(substantive)):
        # Recursive splitting can isolate a heading, so check once more.
        if is_heading_only(chunk.page_content, chunk.metadata):
            removed["heading_only"] += 1
            continue
        # Normalize whitespace and citation markers before comparing body text.
        text = clean_chunk_text(chunk.page_content)
        if not text:
            continue
        # Deduplicate normalized body text within each movie to retain provenance.
        if text in seen:
            removed["duplicates"] += 1
            continue
        seen.add(text)
        # Add the movie title and available headings to every embedded chunk.
        prefix = [f"Article: {title}"]
        for key, label in [("header2", "Section"), ("header3", "Subsection")]:
            if chunk.metadata.get(key):
                prefix.append(f"{label}: {chunk.metadata[key]}")
        # Prepare Chroma fields: source-position ID, contextual text, and metadata.
        items.append({
            "id": f"html-{file_name}-{chunk_index}",
            "document": "\n".join(prefix) + "\n\n" + text,
            "metadata": {**chunk.metadata, "source": file_name, "movie_title": title},
        })
    # Return prepared records and counters; database insertion happens separately.
    return items, removed


### Prepare corpus in memory (no database writes)

In [15]:
all_chunks = []
removed_totals = {"heading_only": 0, "duplicates": 0}
for file_name in sorted(files):
    items, removed = build_movie_chunks(file_name)
    all_chunks.extend(items)
    for key in removed_totals:
        removed_totals[key] += removed[key]
print(f"Prepared {len(all_chunks)} chunks; removed {removed_totals}")
assert all(item["document"].startswith("Article: " + item["metadata"]["movie_title"])
           for item in all_chunks)


Prepared 1022 chunks; removed {'heading_only': 337, 'duplicates': 0}


In [16]:
print(f"Prepared {len(all_chunks)} HTML chunks; ready to index")

Prepared 1022 HTML chunks; ready to index


In [17]:
# add() does not replace old chunks; use a fresh collection.
assert cosine_collection.count() == 0, "Use a fresh collection for corrected chunks."
cosine_collection.add(
    ids=[item["id"] for item in all_chunks],
    documents=[item["document"] for item in all_chunks],
    metadatas=[item["metadata"] for item in all_chunks],
)

In [18]:
repo_dir = Path(path).resolve().parent
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
from rag_design.movie_retrieval import query_movie_documents
titles = sorted({item["metadata"]["movie_title"] for item in all_chunks})
query_movie_documents(cosine_collection,
    "What year and genre describe 8½, and who directed it?", titles)


['Article: 8½\nSection: Influence\n\nLater in the year of the film\'s 1963 release, a group of young Italian writers founded Gruppo \'63, a literary collective of the neoavanguardia composed of novelists, reviewers, critics, and poets inspired by 8½ and Umberto Eco\'s seminal essay, Opera aperta (Open Work). "Imitations of 8½ pile up by directors all over the world", wrote Fellini biographer Tullio Kezich. The following is Kezich\'s short-list of the films it has inspired: Mickey One (Arthur Penn, 1965), Alex in Wonderland (Paul Mazursky, 1970), Beware of a Holy Whore (Rainer Werner Fassbinder, 1971), Day for Night (François Truffaut, 1974), All That Jazz (Bob Fosse, 1979), Stardust Memories (Woody Allen, 1980), Sogni d\'oro (Nanni Moretti, 1981), Planet Parade (Vadim Abdrashitov, 1984), A King and His Movie (Carlos Sorín, 1986), 1993), Living in Oblivion (Tom DiCillo, 1995), 8½ Women (Peter Greenaway, 1999), and 8½ $ (Grigori Konstantinopolsky, 1999). Martin Scorsese included 8½ on hi